# Evolutionary Factor Mining: Mutation + Redundancy + Complexity

This notebook implements three techniques from the QuantaAlpha paper to improve
factor quality beyond one-shot generation:

**1. Complexity constraints** (QuantaAlpha Eq.4): score each factor's structural
complexity (expression length, parameter count, number of raw features). Reject
overly complex factors that are likely to overfit. Their ablation showed removing
this caused the biggest strategy degradation (-8.44% ARR).

**2. AST-based redundancy filtering** (QuantaAlpha Eq.5-6): compute structural
similarity between factor ASTs by normalizing numeric parameters and comparing
subtree sets. Filter near-duplicates that differ only in window sizes (e.g.,
`rank(ts_delta(close, 5))` vs `rank(ts_delta(close, 3))`). Keeps the factor
set diverse and reduces crowding.

**3. Mutation feedback loop**: take low-scoring factors, send their expression +
evaluation metrics back to the LLM with a diagnosis prompt, and generate improved
variants. This closes the generate→evaluate loop and is the core mechanism that
differentiates QuantaAlpha from one-shot generation.

**Pipeline:**
```
Raw factors (1000) → complexity filter → redundancy filter → evaluate
    → select bottom 20% → mutate via LLM → re-evaluate → merge back
    → repeat for N rounds
```

**Requirements:**
- `../results/evaluation/factors_1000.json` (from generate_1000.ipynb)
- `ANTHROPIC_API_KEY` environment variable
- The compute + evaluate functions from opt6_full.ipynb (loaded via data cache)


In [1]:
import os
import re
import json
import ast
import copy
import math
import time
import warnings
import pickle

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
print("Imports ready")


Imports ready


In [2]:
# Load market data
with open("../data/sp500_cache.pkl", "rb") as f:
    data = pickle.load(f)
print(f"Market data: {data['close'].shape[0]} days × {data['close'].shape[1]} stocks")

# Load factors (use 1000-factor set if available, else baseline 216)
factor_path = "../results/evaluation/factors_1000.json"
if not os.path.exists(factor_path):
    factor_path = "../results/evaluation/factors.json"
    print(f"Warning: factors_1000.json not found, falling back to {factor_path}")

with open(factor_path, "r") as f:
    factors_raw = json.load(f)
print(f"Loaded {len(factors_raw)} raw factors from {factor_path}")


Market data: 752 days × 483 stocks
Loaded 573 raw factors from ../results/evaluation/factors_1000.json


## Step 1: Complexity Constraints

In [3]:
ALLOWED_FIELDS = {"open", "high", "low", "close", "volume", "vwap", "returns", "adv20"}

def compute_complexity(expr):
    """
    Factor complexity score inspired by QuantaAlpha Eq.4:
    
        C(f) = α1 · sym_len + α2 · n_params + α3 · log(1 + n_fields)
    
    Where:
      - sym_len: character length of the expression
      - n_params: count of numeric constants (window sizes, thresholds)
      - n_fields: number of distinct raw features used
    
    Higher = more complex = more likely to overfit.
    """
    try:
        tree = ast.parse(expr, mode="eval").body
    except SyntaxError:
        return None
    
    n_params = 0
    fields = set()
    depth = [0]
    
    def walk(node, d=0):
        nonlocal n_params
        depth[0] = max(depth[0], d)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            n_params += 1
        if isinstance(node, ast.Name) and node.id in ALLOWED_FIELDS:
            fields.add(node.id)
        for child in ast.iter_child_nodes(node):
            walk(child, d + 1 if isinstance(node, ast.Call) else d)
    
    walk(tree)
    
    sym_len = len(expr)
    n_fields = len(fields)
    
    # Weighted complexity score (tuned to the distribution of our factor set)
    score = 0.4 * sym_len + 0.3 * n_params + 0.3 * math.log1p(n_fields)
    
    return {
        "complexity": round(score, 2),
        "sym_len": sym_len,
        "n_params": n_params,
        "n_fields": n_fields,
        "depth": depth[0],
    }


# Complexity thresholds (from QuantaAlpha: sym_len ≤ 250, features ≤ 6)
MAX_COMPLEXITY = 80.0   # ~95th percentile of our factor set
MAX_SYM_LEN = 200       # character length
MAX_DEPTH = 5            # nesting depth
MAX_FIELDS = 6           # distinct raw features

def passes_complexity(expr):
    """Check if a factor passes complexity constraints."""
    info = compute_complexity(expr)
    if info is None:
        return False, "parse error", None
    
    reasons = []
    if info["complexity"] > MAX_COMPLEXITY:
        reasons.append(f"complexity {info['complexity']:.0f} > {MAX_COMPLEXITY}")
    if info["sym_len"] > MAX_SYM_LEN:
        reasons.append(f"sym_len {info['sym_len']} > {MAX_SYM_LEN}")
    if info["depth"] > MAX_DEPTH:
        reasons.append(f"depth {info['depth']} > {MAX_DEPTH}")
    if info["n_fields"] > MAX_FIELDS:
        reasons.append(f"fields {info['n_fields']} > {MAX_FIELDS}")
    
    if reasons:
        return False, "; ".join(reasons), info
    return True, "", info


# Apply to all factors
passed = []
rejected_complexity = []

for f in factors_raw:
    expr = f["factor"] if isinstance(f, dict) else f
    ok, reason, info = passes_complexity(expr)
    if ok:
        if isinstance(f, dict):
            f["_complexity"] = info
        passed.append(f)
    else:
        rejected_complexity.append((expr, reason))

print(f"Complexity filter: {len(factors_raw)} → {len(passed)} factors")
print(f"Rejected: {len(rejected_complexity)}")
if rejected_complexity:
    print(f"Example rejections:")
    for expr, reason in rejected_complexity[:5]:
        e = expr if isinstance(expr, str) else expr.get("factor", "?")
        print(f"  {reason}: {e[:60]}")

# Show distribution
complexities = [compute_complexity(f["factor"] if isinstance(f, dict) else f)["complexity"]
                for f in passed if compute_complexity(f["factor"] if isinstance(f, dict) else f)]
print(f"\nComplexity distribution (passed factors):")
print(f"  min={min(complexities):.1f}, p25={np.percentile(complexities,25):.1f}, "
      f"p50={np.percentile(complexities,50):.1f}, p75={np.percentile(complexities,75):.1f}, "
      f"max={max(complexities):.1f}")


Complexity filter: 573 → 573 factors
Rejected: 0

Complexity distribution (passed factors):
  min=7.5, p25=21.0, p50=27.0, p75=33.2, max=72.7


## Step 2: AST-Based Redundancy Filter

In [4]:
def structural_signature(expr):
    """
    Normalize an expression's AST by replacing all numeric constants with 0.
    Two expressions with the same signature differ only in parameter values
    (e.g., window sizes), not in their operator/field structure.
    
    This implements a simplified version of QuantaAlpha Eq.5-6:
    instead of computing the largest common isomorphic subtree (expensive),
    we normalize and compare full-tree signatures. Factors with identical
    signatures are structural duplicates.
    """
    tree = ast.parse(expr, mode="eval").body
    
    def normalize(node):
        node = copy.deepcopy(node)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return ast.Constant(value=0)
        for field, value in ast.iter_fields(node):
            if isinstance(value, ast.AST):
                setattr(node, field, normalize(value))
            elif isinstance(value, list):
                setattr(node, field, [normalize(v) if isinstance(v, ast.AST) else v for v in value])
        return node
    
    return ast.dump(normalize(tree))


def max_common_subtree_size(e1, e2):
    """
    QuantaAlpha Eq.5: size of the largest common normalized subtree.
    
    Used for partial similarity detection — two factors may share a large
    common subexpression even if their overall structure differs.
    Returns the node count of the largest shared subtree.
    """
    def get_subtrees(expr):
        tree = ast.parse(expr, mode="eval").body
        subs = {}
        def walk(node):
            nc = copy.deepcopy(node)
            def norm(n):
                if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)):
                    n.value = 0
                for c in ast.iter_child_nodes(n): norm(c)
            norm(nc)
            s = ast.dump(nc)
            # Count nodes
            size = (s.count("Name(") + s.count("Call(") + s.count("BinOp(") +
                    s.count("UnaryOp(") + s.count("Constant("))
            subs[s] = max(subs.get(s, 0), size)
            for c in ast.iter_child_nodes(node):
                walk(c)
        walk(tree)
        return subs
    
    s1 = get_subtrees(e1)
    s2 = get_subtrees(e2)
    common = set(s1.keys()) & set(s2.keys())
    if not common:
        return 0
    return max(s1[s] for s in common)


# Apply redundancy filter: within each structural group, keep only the first
# (most "canonical") expression.
sig_groups = {}
for f in passed:
    expr = f["factor"] if isinstance(f, dict) else f
    try:
        sig = structural_signature(expr)
    except:
        sig = expr  # fallback: use raw string
    if sig not in sig_groups:
        sig_groups[sig] = []
    sig_groups[sig].append(f)

# Keep one representative per structural group
unique_factors = []
redundant_groups = []
for sig, group in sig_groups.items():
    unique_factors.append(group[0])  # keep first
    if len(group) > 1:
        redundant_groups.append(group)

print(f"Redundancy filter: {len(passed)} → {len(unique_factors)} factors")
print(f"Structural groups with duplicates: {len(redundant_groups)}")
if redundant_groups:
    print(f"\nExample redundant groups (kept first, removed rest):")
    for group in sorted(redundant_groups, key=len, reverse=True)[:5]:
        exprs = [f["factor"] if isinstance(f, dict) else f for f in group]
        print(f"  [{len(group)} factors] Kept: {exprs[0][:60]}")
        for e in exprs[1:3]:
            print(f"    Removed: {e[:60]}")

factors_filtered = unique_factors
print(f"\nFinal filtered set: {len(factors_filtered)} factors")


Redundancy filter: 573 → 561 factors
Structural groups with duplicates: 12

Example redundant groups (kept first, removed rest):
  [2 factors] Kept: rank(ts_std(returns, 5) / ts_std(returns, 20))
    Removed: rank(ts_std(returns, 5) / ts_std(returns, 60))
  [2 factors] Kept: rank(ts_mean(returns, 5)) - rank(ts_mean(returns, 20))
    Removed: rank(ts_mean(returns, 5)) - rank(ts_mean(returns, 60))
  [2 factors] Kept: rank(ts_delta(close, 40)) * rank(ts_delta(volume, 10))
    Removed: rank(ts_delta(close, 10)) * rank(ts_delta(volume, 10))
  [2 factors] Kept: rank(ts_mean(close / vwap, 10))
    Removed: rank(ts_mean(close / vwap, 5))
  [2 factors] Kept: -1 * rank(ts_delta(close, 10) / ts_std(close, 20))
    Removed: -1 * rank(ts_delta(close, 1) / ts_std(close, 10))

Final filtered set: 561 factors


## Step 3: Compute & Evaluate Filtered Factors

In [5]:
# This cell imports the optimized compute and evaluate functions.
# If running standalone, you need the Numba kernels + bottleneck from opt6.
# For simplicity, we use the opt6_full notebook's functions directly.

# Import the backtester and evaluation from opt6_full
# (In practice, you'd run this in the same notebook or import from a module)
# OR: copy the essential functions inline. For this notebook, we'll use
# a simplified compute+evaluate that works standalone:

import bottleneck as bn

# Simplified compute (uses pandas vectorized ops — Opt2 level, fast enough)
class SimpleBacktester:
    def __init__(self, data):
        self.data = data
    def ts_mean(self, x, d):  return x.rolling(d, min_periods=d).mean()
    def ts_std(self, x, d):   return x.rolling(d, min_periods=d).std()
    def ts_delta(self, x, d): return x - x.shift(d)
    def ts_max(self, x, d):   return x.rolling(d, min_periods=d).max()
    def ts_min(self, x, d):   return x.rolling(d, min_periods=d).min()
    def ts_corr(self, x, y, d):
        r = x.rolling(d, min_periods=d).corr(y)
        r = r.replace([float("inf"), float("-inf")], 0.0)
        r.iloc[d-1:] = r.iloc[d-1:].fillna(0.0).values
        return r
    def rank(self, x):   return x.rank(axis=1, pct=True)
    def zscore(self, x):
        mu, sigma = x.mean(axis=1), x.std(axis=1)
        z = x.sub(mu, axis=0).div(sigma, axis=0)
        return z.replace([float("inf"), float("-inf")], 0.0).fillna(0.0)
    def sign(self, x):   return np.sign(x).fillna(0.0)
    def log(self, x):    return np.log(x.clip(lower=1e-9))
    def abs(self, x):    return x.abs()
    def power(self, x, n): return x ** n
    def compute_factor(self, expr):
        ctx = {k: self.data[k] for k in ["open","high","low","close","volume","vwap","returns","adv20"]}
        ctx.update({"ts_mean": self.ts_mean, "ts_std": self.ts_std, "ts_delta": self.ts_delta,
                    "ts_max": self.ts_max, "ts_min": self.ts_min, "ts_corr": self.ts_corr,
                    "rank": self.rank, "zscore": self.zscore, "sign": self.sign,
                    "log": self.log, "abs": self.abs, "power": self.power})
        return eval(expr, {"__builtins__": {}}, ctx)


def compute_all_factors(factors, data):
    bt = SimpleBacktester(data)
    results = {}
    for f in tqdm(factors, desc="Computing factors"):
        expr = f["factor"] if isinstance(f, dict) else f
        try:
            results[expr] = bt.compute_factor(expr)
        except Exception as e:
            pass
    print(f"Computed: {len(results)}/{len(factors)}")
    return results


def evaluate_factor(factor, returns, forward_days=1, quantile=0.1):
    """Vectorized + fused evaluation (from opt6_full)."""
    fwd_returns = returns.shift(-forward_days)
    valid = factor.notna() & fwd_returns.notna()
    n_valid = valid.sum(axis=1)
    f_ranked = factor.where(valid).rank(axis=1, method="first")
    k = (n_valid * quantile).astype(int).clip(lower=1)
    long_thresh = n_valid - k + 1
    short_thresh = k

    # IC
    r_ranked = fwd_returns.rank(axis=1, pct=False).where(valid)
    f_ranked_avg = factor.where(valid).rank(axis=1, pct=False)
    f_mean = f_ranked_avg.mean(axis=1); r_mean = r_ranked.mean(axis=1)
    f_dev = f_ranked_avg.sub(f_mean, axis=0); r_dev = r_ranked.sub(r_mean, axis=0)
    cov = (f_dev * r_dev).sum(axis=1) / (n_valid - 1)
    f_std = f_dev.pow(2).sum(axis=1).div(n_valid - 1).pow(0.5)
    r_std = r_dev.pow(2).sum(axis=1).div(n_valid - 1).pow(0.5)
    ic = (cov / (f_std * r_std)); ic[n_valid < 10] = np.nan; ic = ic.dropna()
    ic_mean = ic.mean(); ic_std = ic.std()
    icir = ic_mean / ic_std if ic_std > 0 else 0.0

    # Long-short
    long_mask = f_ranked.ge(long_thresh, axis=0) & valid
    short_mask = f_ranked.le(short_thresh, axis=0) & valid
    long_ret = (fwd_returns * long_mask).sum(axis=1) / long_mask.sum(axis=1)
    short_ret = (fwd_returns * short_mask).sum(axis=1) / short_mask.sum(axis=1)
    ls_ret = (long_ret - short_ret); ls_ret[n_valid < 20] = np.nan
    ls_ret = ls_ret.iloc[:-forward_days].dropna()
    sharpe = (ls_ret.mean() / ls_ret.std() * np.sqrt(252)) if ls_ret.std() > 0 else 0.0
    nav = (1 + ls_ret).cumprod()
    mdd = ((nav - nav.cummax()) / nav.cummax()).min()

    # Turnover
    holdings = (long_mask | short_mask); holdings[n_valid < 20] = False
    h = holdings.values.astype(np.int8)
    changed = np.abs(h[1:] - h[:-1]).sum(axis=1)
    total = h[1:].sum(axis=1) + h[:-1].sum(axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        daily_to = np.where(total > 0, changed / total, 0.0)
    mask = (h[1:].sum(axis=1) > 0) & (h[:-1].sum(axis=1) > 0)
    turnover = float(np.mean(daily_to[mask])) if mask.any() else np.nan

    return {"IC_mean": round(float(ic_mean), 4), "ICIR": round(float(icir), 4),
            "Sharpe": round(float(sharpe), 4), "MDD": round(float(mdd), 4),
            "Turnover": round(float(turnover), 4)}


def evaluate_all_factors(factor_results, returns):
    rows = []
    for i, (expr, fdf) in enumerate(tqdm(factor_results.items(), desc="Evaluating"), 1):
        metrics = evaluate_factor(fdf, returns)
        metrics["factor"] = expr
        rows.append(metrics)
    df = pd.DataFrame(rows)
    def minmax(s):
        rng = s.max() - s.min()
        return (s - s.min()) / rng if rng > 0 else pd.Series(0.5, index=s.index)
    df["score"] = (minmax(df["ICIR"]) * 0.35 + minmax(df["Sharpe"]) * 0.30 +
                   minmax(df["IC_mean"]) * 0.15 + minmax(df["MDD"]) * 0.15 +
                   minmax(-df["Turnover"]) * 0.05).round(4)
    df = df.sort_values("score", ascending=False).reset_index(drop=True)
    df.index = range(1, len(df) + 1); df.index.name = "rank"
    return df


# Compute and evaluate the filtered set
returns = data["returns"]

t0 = time.time()
factor_results = compute_all_factors(factors_filtered, data)
t_compute = time.time() - t0

t0 = time.time()
results = evaluate_all_factors(factor_results, returns)
t_eval = time.time() - t0

print(f"\nCompute: {t_compute:.1f}s, Evaluate: {t_eval:.1f}s, Total: {t_compute+t_eval:.1f}s")
print(f"\nTop 10 factors:")
print(results.head(10)[["factor", "score", "IC_mean", "ICIR", "Sharpe", "MDD"]].to_string())


Computing factors:   0%|          | 0/561 [00:00<?, ?it/s]

Computed: 561/561


Evaluating:   0%|          | 0/561 [00:00<?, ?it/s]


Compute: 33.5s, Evaluate: 76.3s, Total: 109.8s

Top 10 factors:
                                                                              factor   score  IC_mean    ICIR  Sharpe     MDD
rank                                                                                                                         
1            rank(sign(ts_mean(close, 5) - close) * abs(close - ts_mean(close, 20)))  0.8711   0.0145  0.1011  1.6459 -0.0919
2        rank(ts_max(close, 20) / close) * zscore(log(volume / ts_mean(volume, 15)))  0.8692   0.0123  0.1685  1.0939 -0.0727
3             rank(ts_mean(returns, 15) / ts_max(abs(returns), 15)) * zscore(volume)  0.8654   0.0120  0.0992  1.7755 -0.1231
4                                                          rank(log(volume / adv20))  0.8552   0.0126  0.1757  0.8878 -0.0702
5                           zscore(ts_mean(abs(returns), 10) * rank(volume / adv20))  0.8373   0.0137  0.1262  1.2832 -0.1212
6                                        -1 * zscore(

## Step 4: Evolution Loop — Mutation + Crossover

Two evolutionary operators inspired by QuantaAlpha:

**Mutation (bottom factors):** Take the worst-scoring factors, diagnose why they
failed, and generate improved variants. Targets factors with broken economic
logic — e.g., a momentum signal that's accidentally anti-momentum due to a sign
error, or a vol signal that overfits to a specific regime.

**Crossover (top factors):** Take the best-scoring factors and generate
complementary variants that preserve the core signal but explore adjacent
economic ideas. This is where the real value lies — a factor with Sharpe=1.5
already has a validated economic mechanism; combining it with a volume
confirmation or regime filter can push it higher.

Each round:
1. Crossover the top N factors → generate enhanced variants
2. Mutate the bottom N factors → generate fixed variants  
3. Filter all variants (validate + complexity + redundancy)
4. Compute + evaluate
5. Merge winners back into the pool


In [ ]:
import anthropic

ANTHROPIC_API_KEY = "Your_API_Key_Here"
if not ANTHROPIC_API_KEY:
    raise RuntimeError("ANTHROPIC_API_KEY not set")
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


SYSTEM_PROMPT = """You are a quantitative researcher evolving Alpha factors for stock markets.

Available operators:
  ts_mean(x, d), ts_std(x, d), ts_corr(x, y, d), ts_delta(x, d),
  ts_max(x, d), ts_min(x, d), rank(x), zscore(x),
  sign(x), log(x), abs(x), power(x, n)
  Arithmetic: + - * /

Available data fields:
  open, high, low, close, volume, vwap, returns, adv20

Rules:
- Output a JSON array only, no extra text
- Each item must have "factor" (expression) and "description" (economic rationale)
- Only use the operators and fields listed above
- Keep expressions under 150 characters
- Use at most 4 different data fields per factor
"""


ALLOWED_OPS = {"ts_mean","ts_std","ts_corr","ts_delta","ts_max","ts_min",
               "rank","zscore","sign","log","abs","power"}

def validate_factor(expr):
    try:
        tree = ast.parse(expr, mode="eval")
    except SyntaxError:
        return False
    errors = []
    class Checker(ast.NodeVisitor):
        def visit_Call(self, node):
            if isinstance(node.func, ast.Name) and node.func.id not in ALLOWED_OPS:
                errors.append(True)
            self.generic_visit(node)
        def visit_Name(self, node):
            if node.id not in ALLOWED_FIELDS and node.id not in ALLOWED_OPS:
                errors.append(True)
        def visit_Attribute(self, node):
            errors.append(True)
    Checker().visit(tree)
    return len(errors) == 0


def _filter_new_factors(batch, parent_expr, seen, tag, all_existing_sigs=None):
    """
    Shared post-processing for both mutation and crossover outputs.
    Validates, deduplicates, checks complexity and structural redundancy.
    """
    new_factors = []
    for f in batch:
        new_expr = f.get("factor", "").strip()
        if not new_expr or new_expr in seen or new_expr == parent_expr:
            continue
        if not validate_factor(new_expr):
            continue
        ok, _, _ = passes_complexity(new_expr)
        if not ok:
            continue
        # Structural redundancy check against parent
        try:
            new_sig = structural_signature(new_expr)
            if parent_expr and new_sig == structural_signature(parent_expr):
                continue
            # Also check against all existing factors if provided
            if all_existing_sigs and new_sig in all_existing_sigs:
                continue
        except:
            pass
        
        seen.add(new_expr)
        f["origin"] = tag
        f["parent"] = parent_expr
        new_factors.append(f)
    return new_factors


def crossover_factors(top_factors, results_df, n_variants=3, all_existing_sigs=None):
    """
    Crossover: take high-scoring factors and generate enhanced variants.
    
    For each top factor, the LLM sees its expression + strong metrics and is
    asked to generate complementary variants that:
      - Preserve the core economic mechanism
      - Add a new dimension (volume confirmation, regime filter, multi-timeframe)
      - Combine ideas from different signal families
    
    For pairs of top factors, the LLM combines their strengths into a new factor.
    """
    new_factors = []
    seen = set()
    exprs = top_factors["factor"].tolist()
    
    # Phase A: enhance each top factor individually
    for _, row in tqdm(top_factors.iterrows(), total=len(top_factors), desc="Crossover (enhance)"):
        expr = row["factor"]
        
        user_msg = f"""This alpha factor performs WELL in backtesting:

Expression: {expr}
Metrics:
  IC (mean): {row['IC_mean']}  (>0.02 is decent, this factor is good)
  ICIR: {row['ICIR']}  (>0.1 is decent)
  Sharpe: {row['Sharpe']}  (>1.0 is good)
  MDD: {row['MDD']}  (closer to 0 is better)
  Composite score: {row['score']}

Generate {n_variants} ENHANCED variants that build on this factor's strength:
1. Add a complementary signal dimension (e.g., volume confirmation, volatility filter, trend quality gate)
2. Try a multi-timeframe version (combine short + long lookback windows)
3. Combine with a different economic mechanism (e.g., if it's momentum-based, add a mean-reversion hedge)

Each variant must be structurally DIFFERENT from the original — not just a parameter tweak.
The goal is to IMPROVE on an already-good factor, not to fix something broken.

Output JSON array only."""

        try:
            response = client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=4096,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_msg}]
            )
            raw = next(b.text for b in response.content if b.type == "text").strip()
            raw = re.sub(r"```(?:json)?|```", "", raw).strip()
            batch = json.loads(raw)
            new_factors.extend(_filter_new_factors(
                batch, expr, seen, "crossover_enhance", all_existing_sigs
            ))
        except Exception as e:
            tqdm.write(f"  Crossover failed for {expr[:40]}: {e}")
        time.sleep(0.3)
    
    # Phase B: combine pairs of top factors
    # Select top 5 pairs with lowest pairwise correlation (most complementary)
    if len(exprs) >= 4:
        pairs = []
        for i in range(min(len(exprs), 8)):
            for j in range(i + 1, min(len(exprs), 8)):
                pairs.append((exprs[i], exprs[j]))
        # Limit to 5 pairs to control API cost
        pairs = pairs[:5]
        
        for e1, e2 in tqdm(pairs, desc="Crossover (combine)"):
            r1 = results_df[results_df["factor"] == e1].iloc[0]
            r2 = results_df[results_df["factor"] == e2].iloc[0]
            
            user_msg = f"""Combine the strengths of these two high-performing alpha factors:

Factor A: {e1}
  IC={r1['IC_mean']}, Sharpe={r1['Sharpe']}, Score={r1['score']}

Factor B: {e2}
  IC={r2['IC_mean']}, Sharpe={r2['Sharpe']}, Score={r2['score']}

Generate {n_variants} NEW factors that combine the economic mechanisms of both A and B.
Each factor should capture elements from BOTH parents — not just be a copy of one.
Possible strategies:
  - Multiply/divide signals from A and B to create interaction terms
  - Use one as a filter/gate and the other as the signal
  - Extract the core sub-expression from each and compose them in a new way

Output JSON array only."""

            try:
                response = client.messages.create(
                    model="claude-sonnet-4-20250514",
                    max_tokens=4096,
                    system=SYSTEM_PROMPT,
                    messages=[{"role": "user", "content": user_msg}]
                )
                raw = next(b.text for b in response.content if b.type == "text").strip()
                raw = re.sub(r"```(?:json)?|```", "", raw).strip()
                batch = json.loads(raw)
                new_factors.extend(_filter_new_factors(
                    batch, f"{e1}+{e2}", seen, "crossover_combine", all_existing_sigs
                ))
            except Exception as e:
                tqdm.write(f"  Combine failed: {e}")
            time.sleep(0.3)
    
    return new_factors


def mutate_factors(bad_factors, results_df, n_variants=3, all_existing_sigs=None):
    """
    Mutation: take low-scoring factors, diagnose failures, generate fixes.
    """
    new_factors = []
    seen = set()
    
    for _, row in tqdm(bad_factors.iterrows(), total=len(bad_factors), desc="Mutation (fix)"):
        expr = row["factor"]
        
        user_msg = f"""This alpha factor scored POORLY in backtesting:

Expression: {expr}
Metrics:
  IC (mean): {row['IC_mean']}  (>0.02 is decent — this is bad)
  ICIR: {row['ICIR']}  (>0.1 is decent)
  Sharpe: {row['Sharpe']}  (>1.0 is good)
  MDD: {row['MDD']}  (closer to 0 is better)
  Composite score: {row['score']}

Diagnose what's wrong with this factor's economic logic, then generate {n_variants} improved variants.
For each variant:
  - Fix the identified weakness (e.g., add risk normalization, flip the sign, change timeframe)
  - Keep the core economic idea but change the implementation substantially
  - Make it structurally DIFFERENT from the original

Output JSON array only."""

        try:
            response = client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=4096,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_msg}]
            )
            raw = next(b.text for b in response.content if b.type == "text").strip()
            raw = re.sub(r"```(?:json)?|```", "", raw).strip()
            batch = json.loads(raw)
            new_factors.extend(_filter_new_factors(
                batch, expr, seen, "mutation", all_existing_sigs
            ))
        except Exception as e:
            tqdm.write(f"  Mutation failed for {expr[:40]}: {e}")
        time.sleep(0.3)
    
    return new_factors


print("Evolution functions ready (crossover + mutation)")


Evolution functions ready (crossover + mutation)


In [7]:
# ── Evolution loop: Crossover (top) + Mutation (bottom) ─────────────────────

N_ROUNDS = 3
TOP_N = 15          # crossover the top 15 factors each round
BOTTOM_N = 15       # mutate the bottom 15 factors each round
N_VARIANTS = 3      # generate 3 variants per factor

all_results = results.copy()
all_factor_results = dict(factor_results)
evolution_log = []

# Build a set of existing structural signatures for redundancy checking
existing_sigs = set()
for _, row in all_results.iterrows():
    try:
        existing_sigs.add(structural_signature(row["factor"]))
    except:
        pass

for round_num in range(N_ROUNDS):
    print(f"\n{'═'*70}")
    print(f"EVOLUTION ROUND {round_num + 1}/{N_ROUNDS}")
    print(f"{'═'*70}")
    print(f"Current pool: {len(all_results)} factors, "
          f"mean score: {all_results['score'].mean():.4f}, "
          f"top score: {all_results['score'].max():.4f}")
    
    round_new_factors = []
    
    # ── Phase A: Crossover (top factors) ─────────────────────────────────
    top_factors = all_results.head(TOP_N)
    print(f"\n▸ Crossover: enhancing top {TOP_N} factors "
          f"(score range: {top_factors['score'].min():.4f} – {top_factors['score'].max():.4f})")
    
    crossover_new = crossover_factors(
        top_factors, all_results, n_variants=N_VARIANTS,
        all_existing_sigs=existing_sigs
    )
    round_new_factors.extend(crossover_new)
    print(f"  → {len(crossover_new)} new factors from crossover")
    
    # ── Phase B: Mutation (bottom factors) ───────────────────────────────
    bottom_factors = all_results.tail(BOTTOM_N)
    print(f"\n▸ Mutation: fixing bottom {BOTTOM_N} factors "
          f"(score range: {bottom_factors['score'].min():.4f} – {bottom_factors['score'].max():.4f})")
    
    mutation_new = mutate_factors(
        bottom_factors, all_results, n_variants=N_VARIANTS,
        all_existing_sigs=existing_sigs
    )
    round_new_factors.extend(mutation_new)
    print(f"  → {len(mutation_new)} new factors from mutation")
    
    if not round_new_factors:
        print("\nNo valid new factors produced, stopping early")
        break
    
    # ── Phase C: Compute + Evaluate new factors ──────────────────────────
    print(f"\n▸ Computing + evaluating {len(round_new_factors)} new factors...")
    new_factor_results = compute_all_factors(round_new_factors, data)
    
    if not new_factor_results:
        print("  No factors computed successfully, skipping round")
        continue
    
    new_eval = evaluate_all_factors(new_factor_results, returns)
    
    # ── Phase D: Merge winners ───────────────────────────────────────────
    threshold = all_results["score"].median()
    good_crossover = [f for f in crossover_new if f["factor"] in new_factor_results 
                      and f["factor"] in new_eval["factor"].values
                      and float(new_eval[new_eval["factor"]==f["factor"]]["score"].iloc[0]) > threshold]
    good_mutation = [f for f in mutation_new if f["factor"] in new_factor_results
                     and f["factor"] in new_eval["factor"].values
                     and float(new_eval[new_eval["factor"]==f["factor"]]["score"].iloc[0]) > threshold]
    
    # Add to pool
    n_added = 0
    for expr in new_factor_results:
        if expr not in all_factor_results:
            if expr in new_eval["factor"].values:
                score = float(new_eval[new_eval["factor"]==expr]["score"].iloc[0])
                if score > threshold:
                    all_factor_results[expr] = new_factor_results[expr]
                    try:
                        existing_sigs.add(structural_signature(expr))
                    except:
                        pass
                    n_added += 1
    
    print(f"\n▸ Merged {n_added} factors above median threshold ({threshold:.4f})")
    
    # Re-evaluate full pool
    all_results = evaluate_all_factors(all_factor_results, returns)
    
    # Log
    round_stats = {
        "round": round_num + 1,
        "crossover_generated": len(crossover_new),
        "mutation_generated": len(mutation_new),
        "total_new": len(round_new_factors),
        "added_to_pool": n_added,
        "pool_size": len(all_results),
        "pool_mean_score": round(all_results["score"].mean(), 4),
        "pool_max_score": round(all_results["score"].max(), 4),
        "crossover_above_median": len(good_crossover),
        "mutation_above_median": len(good_mutation),
    }
    evolution_log.append(round_stats)
    
    print(f"\n  Round {round_num+1} summary:")
    print(f"    Crossover: {len(crossover_new)} generated → {len(good_crossover)} above median")
    print(f"    Mutation:  {len(mutation_new)} generated → {len(good_mutation)} above median")
    print(f"    Pool: {len(all_results)} factors, mean={round_stats['pool_mean_score']:.4f}, "
          f"max={round_stats['pool_max_score']:.4f}")

print(f"\n{'═'*70}")
print(f"EVOLUTION COMPLETE")
print(f"{'═'*70}")
print(f"Final pool: {len(all_results)} factors")
print(f"\nEvolution log:")
print(f"  {'Round':>5} {'Cross':>6} {'Mut':>5} {'Added':>6} {'Pool':>5} {'Mean':>7} {'Max':>7} {'Cross>med':>10} {'Mut>med':>8}")
for e in evolution_log:
    print(f"  {e['round']:>5} {e['crossover_generated']:>6} {e['mutation_generated']:>5} "
          f"{e['added_to_pool']:>6} {e['pool_size']:>5} {e['pool_mean_score']:>7.4f} "
          f"{e['pool_max_score']:>7.4f} {e['crossover_above_median']:>10} {e['mutation_above_median']:>8}")



══════════════════════════════════════════════════════════════════════
EVOLUTION ROUND 1/3
══════════════════════════════════════════════════════════════════════
Current pool: 561 factors, mean score: 0.5169, top score: 0.8711

▸ Crossover: enhancing top 15 factors (score range: 0.7984 – 0.8711)


Crossover (enhance):   0%|          | 0/15 [00:00<?, ?it/s]

Crossover (combine):   0%|          | 0/5 [00:00<?, ?it/s]

  → 60 new factors from crossover

▸ Mutation: fixing bottom 15 factors (score range: 0.0662 – 0.2149)


Mutation (fix):   0%|          | 0/15 [00:00<?, ?it/s]

...

Computed: 82/82


Evaluating:   0%|          | 0/82 [00:00<?, ?it/s]


▸ Merged 51 factors above median threshold (0.5534)


Evaluating:   0%|          | 0/733 [00:00<?, ?it/s]


  Round 3 summary:
    Crossover: 41 generated → 37 above median
    Mutation:  41 generated → 14 above median
    Pool: 733 factors, mean=0.5420, max=0.9376

══════════════════════════════════════════════════════════════════════
EVOLUTION COMPLETE
══════════════════════════════════════════════════════════════════════
Final pool: 733 factors

Evolution log:
  Round  Cross   Mut  Added  Pool    Mean     Max  Cross>med  Mut>med
      1     60    42     58   619  0.5157  0.8819         47       11
      2     53    42     63   682  0.5267  0.9058         46       17
      3     41    41     51   733  0.5420  0.9376         37       14


## Step 5: Results Analysis

In [8]:
# Compare pre-evolution vs post-evolution factor quality

print("Top 20 factors after evolution:")
print(all_results.head(20)[["factor", "score", "IC_mean", "ICIR", "Sharpe", "MDD", "Turnover"]].to_string())

# Check origins of top 20 factors
top20_exprs = set(all_results.head(20)["factor"].tolist())

# Count by origin
origin_counts = {"original": 0, "crossover_enhance": 0, "crossover_combine": 0, "mutation": 0}
for round_factors in evolution_log:
    pass  # stats already in log

# Check against all new factors generated
all_new = []
for e in evolution_log:
    pass

# Score distribution comparison
print(f"\nScore distribution:")
print(f"  Before evolution ({len(results)} factors):")
print(f"    mean={results['score'].mean():.4f}, median={results['score'].median():.4f}, "
      f"max={results['score'].max():.4f}")
print(f"  After evolution ({len(all_results)} factors):")
print(f"    mean={all_results['score'].mean():.4f}, median={all_results['score'].median():.4f}, "
      f"max={all_results['score'].max():.4f}")

# Save final results
output_path = "../results/evaluation/factors_evolved.json"
evolved_factors = []
for _, row in all_results.iterrows():
    evolved_factors.append({
        "factor": row["factor"],
        "score": float(row["score"]),
        "IC_mean": float(row["IC_mean"]),
        "ICIR": float(row["ICIR"]),
        "Sharpe": float(row["Sharpe"]),
        "MDD": float(row["MDD"]),
        "Turnover": float(row["Turnover"]),
    })

with open(output_path, "w") as f:
    json.dump(evolved_factors, f, indent=2)
print(f"\nSaved {len(evolved_factors)} evolved factors to {output_path}")

# Evolution efficiency: crossover vs mutation hit rate
print(f"\nEvolution efficiency:")
for e in evolution_log:
    cross_rate = e["crossover_above_median"] / max(e["crossover_generated"], 1) * 100
    mut_rate = e["mutation_above_median"] / max(e["mutation_generated"], 1) * 100
    print(f"  Round {e['round']}: crossover hit rate = {cross_rate:.0f}%, "
          f"mutation hit rate = {mut_rate:.0f}%")


Top 20 factors after evolution:
                                                                                                                                                                                   factor   score  IC_mean    ICIR  Sharpe     MDD  Turnover
rank                                                                                                                                                                                                                                        
1        rank(volume / adv20) * rank(sign(ts_mean(close, 5) - close) * abs(close - ts_mean(close, 20))) * rank(ts_corr(volume, abs(returns), 15)) / (1 + ts_std(log(volume / adv20), 10))  0.9376   0.0186  0.1863  2.2255 -0.0795    0.5022
2                          rank(vwap / close - 1) * power(rank(volume / adv20), 2) / (1 + ts_std(returns, 5) * ts_std(log(volume / adv20), 10)) * rank(ts_corr(volume, abs(returns), 20))  0.9038   0.0187  0.2029  1.9468 -0.1441    0.6595
3                   